You cannot run this like a normal notebook cell ❌
This must run inside a DLT pipeline

In [ ]:
import dlt
from pyspark.sql.functions import *

1. Create your bronze layer tables ingesting from the landing zone

In [ ]:
#here we are creating a customer_raw table and loading the data from landing zone
#every functions should return a dataframe 
@dlt.table(name = "customer_raw") # to create a managed table and input will be from the function below
def customer_raw():
  return (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .load("/Volumes/dbacademy/labuser14488647_1776864670/landing_zone/customers/") #customer data
    .withColumn("load_time", current_timestamp())
  )

#Whatever this function returns = data for the  customer_raw table

In [ ]:
#here we are creating a invoice_raw table and loading the data from landing zone
@dlt.table(name = "invoice_raw") # to create a table
def invoice_raw():
  return (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .load("/Volumes/dbacademy/labuser14488647_1776864670/landing_zone/invoices/") #invoice data
    .withColumn("load_time", current_timestamp())
  )

2. Create your silver layer tables reading incremental data from bronze layer

In [ ]:
#here we are creating a customer_cleaned table and loading the data from customer_raw table -- only incremental load as we using readStream
@dlt.table(name = "customer_cleaned") 
@dlt.expect_or_drop("valid_customer", "customer_id IS NOT NULL") #for validation-- Keep only records where customer_id is not null; drop the rest. valid_customer = validation name
def get_customer_clean():
  return (
    spark.readStream
    .format("delta")
    .table("live.customer_raw") 
    .selectExpr("CustomerID as customer_id", "CustomerName as customer_name", "load_time"))
    

In [ ]:
#dlt.table is a function where we can define table name partition_col and other parameters to create a table.. partition_cols-- define the partition columns
@dlt.table(name = "invoice_cleaned", partition_cols=["invoice_year","Country"])
@dlt.expect_or_drop("valid_invoice_and_qty", "invoice_no IS NOT NULL AND quantity > 0")
def get_invoice_clean():
  return (
    spark.readStream
    .format("delta")
    .table("live.invoice_raw")
    .selectExpr("InvoiceNo as invoice_no", "StockCode as stock_code", "Description as description","Quantity as quantity", "to_date(InvoiceDate, 'd-M-y H.m') as invoice_date","UnitPrice as unit_price", "CustomerID as customer_id", "Country as country","year(to_date(InvoiceDate, 'd-M-y H.m')) as invoice_year","month(to_date(InvoiceDate, 'd-M-y H.m')) as invoice_month", "load_time")
  ) 


3. Build your SCD Type 2 dimensions using CDC from silver layer

In [ ]:
dlt.create_streaming_table("customers") #create streaming table
# It updates the customers table by tracking changes and keeping history (SCD Type 2)
dlt.apply_changes(
    target = "customers", #Final table where data will be stored
    source = "customers_cleaned",
    keys = ["customer_id"], #Unique identifier (used to match records)
    sequence_by = col("load_time"), #Decides latest record based on timestamp
    stored_as_scd_type = 2 #Apply SCD Type 2 logic --> i.e Keep old records and Insert new records when data changes
)

4. Merge into your fact table using CDC from the silver layer

In [ ]:
dlt.create_streaming_table("invoices",partition_cols=["invoice_year","Country"])

dlt.apply_changes(
    target = "invoices", #Final table where data will be stored
    source = "invoice_cleaned",
    keys = ["invoice_no","stock_code","invoice_date"], #Unique identifier (used to match records)
    sequence_by = col("load_time"), #Decides latest record based on timestamp
)

5. Materialize your gold layer summary using silver layer fact

In [ ]:
@dlt.table(name = "daily_sales_uk_2022")
def compute_daily_sales_uk_2022():
    return( spark.read
                .format(delta)
                .table("live.invoices")
                .where("invoice_year = 2022 AND country = 'United Kingdom'")
                .groupBy("country", "invoice_year", "invoice_month", "invoice_date")
                .agg(expr("round(sum(quantity*unit_price),2)").alias("total_sales"))
                )

after this we need to a pipeline -- >
1. to run the pipeline we need to multi node + share cluster
2. above while creating a table we have not mentioned the schema, where we need to mention it while creating a pipeline
3. inputs to run this scripts is available in dlt_inputs folder